first run in the terminal:

`ngrok http --url=studied-yong-unstroked.ngrok-free.dev 8888`

This notebook starts a local callback server on `127.0.0.1:8888` and lets ngrok forward the HTTPS callback to it. That avoids the manual copy-paste step.

In [1]:
# --- Cell 2: auth + fetch top 5 liked songs ---
import os
import threading
import time
import urllib.parse
import webbrowser
from http.server import BaseHTTPRequestHandler, HTTPServer

import spotipy
from spotipy.oauth2 import SpotifyOAuth

def _load_dotenv(path=".env"):
    try:
        with open(path, "r", encoding="utf-8") as f:
            for raw_line in f:
                line = raw_line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip('"').strip("'")
                if key and key not in os.environ:
                    os.environ[key] = value
    except FileNotFoundError:
        return

_load_dotenv()

CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")
REDIRECT_URI = "https://studied-yong-unstroked.ngrok-free.dev/callback"  # from your screenshot

SCOPE = "user-library-read"

missing = []
if not CLIENT_ID:
    missing.append("SPOTIFY_CLIENT_ID")
if not CLIENT_SECRET:
    missing.append("SPOTIFY_CLIENT_SECRET")
if missing:
    raise ValueError("Missing environment variable(s): " + ", ".join(missing))

auth_manager = SpotifyOAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    redirect_uri=REDIRECT_URI,
    scope=SCOPE,
    open_browser=False,
    cache_path=".spotify_cache",
)

code_holder = {}

class CallbackHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        params = urllib.parse.parse_qs(urllib.parse.urlparse(self.path).query)
        code_holder["code"] = params.get("code", [None])[0]
        self.send_response(200)
        self.end_headers()
        self.wfile.write(b"You can close this tab.")
    def log_message(self, format, *args):
        return

server = HTTPServer(("127.0.0.1", 8888), CallbackHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()

webbrowser.open(auth_manager.get_authorize_url())

while "code" not in code_holder:
    time.sleep(0.2)

token_info = auth_manager.get_access_token(code_holder["code"])
server.shutdown()

sp = spotipy.Spotify(auth=token_info["access_token"])

results = sp.current_user_saved_tracks(limit=5)

print("Top 5 Liked Songs (most recently liked):\n")
for i, item in enumerate(results["items"], start=1):
    track = item["track"]
    name = track["name"]
    artists = ", ".join(a["name"] for a in track["artists"])
    album = track["album"]["name"]
    url = track["external_urls"]["spotify"]
    added_at = item.get("added_at")
    print(f"{i}. {name} — {artists} | {album}")
    print(f"   Added at: {added_at}")
    print(f"   {url}\n")

/var/folders/90/v3nrhyt10tndqn2m9dzj3hnc0000gr/T/ipykernel_51680/3164870207.py:72: DeprecationWarning: You're using 'as_dict = True'.get_access_token will return the token string directly in future versions. Please adjust your code accordingly, or use get_cached_token instead.
  token_info = auth_manager.get_access_token(code_holder["code"])


Top 5 Liked Songs (most recently liked):

1. Numba1 (SSBB Mix) — Samurai Breaks | SSBB008
   Added at: 2026-02-28T13:08:13Z
   https://open.spotify.com/track/2KmvgWDtfBlVclC5hiJ4HR

2. 101 After Dark — Exos | Sweet Dreams
   Added at: 2026-02-28T12:01:09Z
   https://open.spotify.com/track/1ROFqEER1GBxieQ7Glcsex

3. Greed — Kercha | Chase Me
   Added at: 2026-02-28T09:55:29Z
   https://open.spotify.com/track/6Ec4iwtBuyOhYBbJeHVGyI

4. Deaths Effect - Mixed — Julz Da Deejay | Fabriclive 100: Kode9 & Burial (DJ Mix)
   Added at: 2026-02-25T21:14:19Z
   https://open.spotify.com/track/2I6WnLBdWtJ9lxNWo1oI6n

5. Fantasy — Miles Davis (With Easy Mo Bee) | Doo-Bop
   Added at: 2026-02-25T14:26:40Z
   https://open.spotify.com/track/3Yf7Es6k9IDUh5wUczd9r7

